# MVA v3 · NB04C2 — AlphaGenome L737* splice-junction deconvolution
**v1.0.0 · public-variant-only · research use only**

## Why this notebook exists

NB04C found:

- `L737*`: `RNA_LEVEL_POSITIVE`
  - 92 extreme primary rows
  - all 92 in a single modality: `SPLICE_JUNCTIONS`
  - max absolute quantile = 0.99615246
- `N1002K`: `RNA_LEVEL_NOT_SUPPORTED`
  - 0 extreme primary rows
  - max absolute primary quantile = 0.98619723

The AlphaGenome `SpliceJunctionScorer` is non-directional and reports the maximum absolute change in predicted
junction-spanning RNA-seq counts. Therefore the existing aggregate result does **not** tell us which BUB1B
junction drives the score.

This notebook identifies the affected junction coordinate(s) and cross-biosample reproducibility so that a
targeted RT-PCR/junction assay can be designed.

No private VCF, genotype, sample identifier, or additional patient variant is used.

## Frozen interpretation

A junction is `TARGET_JUNCTION` if:

1. it has at least one row with `abs(quantile_score) >= 0.995`; and
2. it is extreme in >=10 distinct tracks/biosamples.

This is a research hypothesis-generation rule only.

AlphaGenome output is not used as ACMG PS3 evidence and is not a second PP3 vote.

In [ ]:
import os
import importlib.util, subprocess, sys, os, json
if importlib.util.find_spec("alphagenome") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","-U","alphagenome"])

import numpy as np, pandas as pd
from pathlib import Path
from alphagenome.data import genome
from alphagenome.models import dna_client, variant_scorers

API_KEY=None
try:
    from google.colab import userdata
    API_KEY=userdata.get("ALPHA_GENOME_API_KEY")
except Exception:
    API_KEY=os.environ.get("ALPHA_GENOME_API_KEY")
assert API_KEY, "Set ALPHA_GENOME_API_KEY in Colab Secrets/environment."

model=dna_client.create(
    api_key=API_KEY,
    address="dns:///gdmscience.googleapis.com:443",
)

from google.colab import drive
drive.mount("/content/drive",force_remount=False)

ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
PUBLIC_OUT=ROOT/"03_PUBLIC_ARTIFACTS"
DOC_OUT=ROOT/"04_DOCUMENTATION"
PUBLIC_OUT.mkdir(parents=True,exist_ok=True)
DOC_OUT.mkdir(parents=True,exist_ok=True)

L737 = genome.Variant(
    chromosome="chr15",
    position=40209701,
    reference_bases="T",
    alternate_bases="G",
)
interval=L737.reference_interval.resize(dna_client.SEQUENCE_LENGTH_1MB)
print("Variant:",L737)

## 1. Re-score only BUB1B splice junctions

In [ ]:
scorer=variant_scorers.RECOMMENDED_VARIANT_SCORERS["SPLICE_JUNCTIONS"]

scores=model.score_variant(
    interval=interval,
    variant=L737,
    variant_scorers=[scorer],
)

tidy=variant_scorers.tidy_scores(
    scores,
    match_gene_strand=True,
    include_extended_metadata=True,
)
assert tidy is not None and len(tidy)>0

bub=tidy[tidy["gene_name"].fillna("").str.upper().eq("BUB1B")].copy()
assert len(bub)>0, "No BUB1B splice-junction rows returned."

required=["junction_Start","junction_End","raw_score","quantile_score"]
for c in required:
    assert c in bub.columns, f"Missing required column: {c}"

bub["abs_quantile"]=bub["quantile_score"].abs()
bub["extreme"]=bub["abs_quantile"]>=0.995

print("BUB1B junction-score rows:",len(bub))
print("Extreme rows:",int(bub.extreme.sum()))

## 2. Identify the junction(s) driving the signal

In [ ]:
track_cols=[c for c in ["track_name","biosample_name","ontology_curie","gtex_tissue"] if c in bub.columns]

def nunique_tracks(g):
    if not track_cols:
        return len(g)
    return g[track_cols].astype(str).drop_duplicates().shape[0]

rows=[]
for (js,je),g in bub.groupby(["junction_Start","junction_End"],dropna=False):
    gx=g[g.extreme].copy()
    rows.append({
        "junction_Start":js,
        "junction_End":je,
        "n_rows":len(g),
        "n_extreme_rows":len(gx),
        "n_extreme_tracks":nunique_tracks(gx) if len(gx) else 0,
        "max_abs_quantile":float(g.abs_quantile.max()),
        "median_raw_score":float(g.raw_score.median()),
        "max_raw_score":float(g.raw_score.max()),
    })

junc=pd.DataFrame(rows).sort_values(
    ["n_extreme_tracks","max_abs_quantile"],
    ascending=[False,False]
).reset_index(drop=True)

junc["call"]=np.where(
    (junc.n_extreme_tracks>=10)&(junc.max_abs_quantile>=0.995),
    "TARGET_JUNCTION",
    "NOT_TARGET"
)

display(junc.head(30))
junc.to_csv(PUBLIC_OUT/"NB04C2_L737STAR_JUNCTION_SUMMARY.csv",index=False)

## 3. Save the cross-biosample target evidence

In [ ]:
targets=junc[junc.call=="TARGET_JUNCTION"].copy()
assert len(targets)>0, (
    "No junction met the frozen cross-biosample rule. "
    "Close the junction-specific follow-up as unsupported."
)

target_pairs=set(zip(targets.junction_Start,targets.junction_End))
top=bub[
    bub.apply(lambda r:(r.junction_Start,r.junction_End) in target_pairs,axis=1)
].copy()

keep=[
    c for c in [
        "junction_Start","junction_End","track_name","biosample_name",
        "gtex_tissue","ontology_curie","raw_score","quantile_score",
        "abs_quantile","extreme"
    ] if c in top.columns
]
top[keep].sort_values("abs_quantile",ascending=False).to_csv(
    PUBLIC_OUT/"NB04C2_L737STAR_TARGET_JUNCTION_TRACKS.csv",
    index=False
)

display(top[keep].sort_values("abs_quantile",ascending=False).head(40))

## 4. Experimental design handoff

In [ ]:
from datetime import datetime, timezone

target_records=targets[
    ["junction_Start","junction_End","n_extreme_tracks","max_abs_quantile",
     "median_raw_score","max_raw_score"]
].to_dict(orient="records")

handoff={
    "schema":"MVA-NB04C2-L737STAR-JUNCTION-DECONVOLUTION-1.0",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "status":"COMPLETE",
    "variant":"BUB1B L737* / chr15:40209701:T>G",
    "reason":"NB04C produced 92 extreme SPLICE_JUNCTIONS rows across biosamples.",
    "target_junctions":target_records,
    "directionality":"The recommended SpliceJunctionScorer is non-directional; REF->ALT direction must be measured experimentally or obtained from raw REF/ALT prediction tracks.",
    "recommended_experiment":{
        "assay":"targeted RT-PCR / junction-specific qPCR in the isogenic Stage-F0 system",
        "minimum_controls":[
            "WT/WT",
            "L737*/WT",
            "L737*/N1002K",
            "WT BUB1B rescue where applicable"
        ],
        "readouts":[
            "canonical junction abundance",
            "candidate alternative junction abundance",
            "total BUB1B transcript abundance",
            "allele-aware transcript abundance if technically feasible"
        ],
        "interpretation":[
            "junction shift + lower transcript supports an RNA-processing contribution to the L737* null allele",
            "no junction shift leaves canonical NMD as the primary L737* loss-of-function mechanism"
        ]
    },
    "acmg_rule":{
        "alphagenome_is_PS3":False,
        "use_for_clinvar_classification":False,
        "use":"research-only experiment prioritization"
    },
    "privacy":{
        "private_patient_data_sent":False,
        "only_already_public_variant_sent":True
    },
    "terms_notice":"AlphaGenome outputs are research-only and subject to the AlphaGenome Output Terms of Use; they are not used for clinical decision-making."
}

(DOC_OUT/"NB04C2_L737STAR_JUNCTION_HANDOFF.json").write_text(
    json.dumps(handoff,indent=2)
)
print(json.dumps(handoff,indent=2))